# CineScope V2 — Basic EDA Notebook
### IMDb Top 10,000 Movies + Personal Viewing Analysis

**Assignment:** Exploratory Data Analysis using NumPy and Pandas  
**Libraries used:** `numpy`, `pandas` (stdlib only)  
**Datasets:**
- `Top_10000_Movies_IMDb.csv` — 10,000 highest-rated IMDb movies
- `Watched.csv` — Personal IMDb ratings export

**Workflow:** Load → Inspect → Clean → Analyse → Personal Data → Merge → Conclusions

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
# Only numpy and pandas are used — no seaborn, no matplotlib, no sklearn
import numpy as np
import pandas as pd
import ast, re
from collections import Counter

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.2f}'.format)
print(f'NumPy  {np.__version__}')
print(f'Pandas {pd.__version__}')

## 1. Loading and Inspecting the Data

We start with `pd.read_csv()` and immediately check shape, column names, and data types. This is always Step 1 in any EDA.

In [ ]:
# ── Load master dataset ──────────────────────────────────────────────────────
df = pd.read_csv('Top_10000_Movies_IMDb.csv', encoding='utf-8', on_bad_lines='skip')
print(f'Shape: {df.shape}  ({df.shape[0]:,} rows × {df.shape[1]} columns)')
df.head(3)

In [ ]:
# Column types and non-null counts
df.info()

In [ ]:
# Statistical summary of numeric columns
df.describe()

## 2. Handling Missing Values

Before any analysis we must know *where* the gaps are and decide:
- **Drop the row** — if a key field (Rating, Votes) is missing
- **Fill with median/mode** — for optional enrichment fields (Metascore, Gross)
- **Leave as NaN** — and handle it column-by-column during analysis

In [ ]:
# ── Missing value audit ──────────────────────────────────────────────────────
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)

audit = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
audit = audit[audit['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print(audit)
print(f'\nTotal rows before cleaning: {len(df):,}')

## 3. Data Cleaning

The raw CSV has several columns that need parsing before use:
- **Runtime** is stored as `"142 min"` → extract the integer with regex
- **Directors** is a Python-list string `["Name", ...]` → parse with `ast.literal_eval`
- **Genre** is a comma-separated string `"Crime, Drama"` → split into a list
- **Link** contains the IMDb tconst `tt0111161` → extract with regex

In [ ]:
# ── Parse Runtime ────────────────────────────────────────────────────────────
df['runtime_mins'] = (df['Runtime']
                      .str.extract(r'(\d+)', expand=False)
                      .pipe(pd.to_numeric, errors='coerce'))

# ── Parse numeric fields ──────────────────────────────────────────────────────
df['rating'] = pd.to_numeric(df['Rating'],    errors='coerce')
df['votes']  = pd.to_numeric(df['Votes'],     errors='coerce')
df['gross']  = pd.to_numeric(df['Gross'],     errors='coerce')
df['meta']   = pd.to_numeric(df['Metascore'], errors='coerce')

# ── Extract tconst from URL ───────────────────────────────────────────────────
df['tconst'] = df['Link'].str.extract(r'(tt\d+)', expand=False)

# ── Parse Directors list (Python-style string) ───────────────────────────────
def parse_list(s):
    try:   return ast.literal_eval(s) if isinstance(s, str) else []
    except: return []

df['dirs_list']  = df['Directors'].apply(parse_list)
df['stars_list'] = df['Stars'].apply(parse_list)

# Director = first person in dirs_list NOT in stars_list
def extract_director(row):
    stars = set(row['stars_list'])
    for name in row['dirs_list']:
        if name not in stars:
            return name
    return row['dirs_list'][0] if row['dirs_list'] else ''

df['director'] = df.apply(extract_director, axis=1)

# ── Parse Genres ──────────────────────────────────────────────────────────────
df['genres_list'] = df['Genre'].str.split(r',\s*')

# ── Drop rows missing critical fields ────────────────────────────────────────
df_clean = df.dropna(subset=['rating', 'tconst']).copy()
df_clean = df_clean[df_clean['rating'].between(1.0, 10.0)]
df_clean = df_clean.drop_duplicates(subset=['tconst'])
print(f'Rows after cleaning: {len(df_clean):,}')

## 4. Rating Distribution

Using `numpy.histogram` to bin ratings into 0.5-wide buckets. This reveals whether the data is normally distributed or skewed.

In [ ]:
# ── Rating distribution with numpy ───────────────────────────────────────────
ratings       = df_clean['rating'].dropna().values
bins          = np.arange(1.0, 10.5, 0.5)
counts, edges = np.histogram(ratings, bins=bins)

print('=== RATING DISTRIBUTION ===')
print(f"{'Bin':<10} {'Count':>8}  Bar")
for lo, hi, c in zip(edges[:-1], edges[1:], counts):
    bar = '█' * (c // 50)
    print(f'{lo:.1f}–{hi:.1f}   {c:>8,}  {bar}')

print(f'\n── Summary Stats ──')
print(f'Mean:          {np.mean(ratings):.2f}')
print(f'Median:        {np.median(ratings):.2f}')
print(f'Std dev:       {np.std(ratings):.2f}')
print(f'% above 7.0:   {(ratings >= 7.0).mean()*100:.1f}%')
print(f'% above 8.0:   {(ratings >= 8.0).mean()*100:.1f}%')
print(f'% below 5.0:   {(ratings < 5.0).mean()*100:.1f}%')

## 5. Genre Analysis

`Genre` is a multi-value column (a movie can be `"Drama, Crime"`). We use `Counter` accumulation — iterating row by row — to count each genre tag without double-counting rows.

In [ ]:
# ── Genre count + average rating per genre ───────────────────────────────────
TARGET_GENRES = ['Drama','Comedy','Action','Thriller','Romance','Crime',
                 'Adventure','Sci-Fi','Horror','Biography','Animation','Mystery']

genre_ratings = {g: [] for g in TARGET_GENRES}
genre_count   = Counter()

for _, row in df_clean.iterrows():
    for g in row['genres_list']:
        genre_count[g] += 1
        if g in genre_ratings:
            genre_ratings[g].append(row['rating'])

print(f"{'Genre':<15} {'Count':>8} {'Avg':>8} {'Median':>8} {'Std':>6}")
print('-' * 55)
for g in sorted(TARGET_GENRES, key=lambda x: -genre_count[x]):
    r = np.array(genre_ratings[g])
    if len(r) == 0: continue
    print(f"{g:<15} {genre_count[g]:>8,} {np.mean(r):>8.2f} {np.median(r):>8.2f} {np.std(r):>6.2f}")

## 6. Top 10 Highest-Rated Movies

Filter, sort descending on `rating`, take first 10 rows.

In [ ]:
# ── Top 10 highest-rated ─────────────────────────────────────────────────────
top10 = (df_clean
         .sort_values('rating', ascending=False)
         [['Movie Name','rating','votes','director','Genre']]
         .head(10)
         .reset_index(drop=True))
top10.index += 1
top10.columns = ['Movie', 'Rating', 'Votes', 'Director', 'Genre']
print(top10.to_string())

## 7. Most Prolific Directors & Their Average Rating

`groupby` + manual iteration with `numpy` lets us compute any statistic we need, even std deviation across groups.

In [ ]:
# ── Director stats (≥ 3 films) ───────────────────────────────────────────────
dir_stats = []
for director, grp in df_clean.groupby('director'):
    if len(grp) < 3 or not director: continue
    r = grp['rating'].dropna().values
    dir_stats.append({
        'Director': director,  'Films': len(grp),
        'Avg': round(float(np.mean(r)), 2),
        'Median': round(float(np.median(r)), 2),
        'Std': round(float(np.std(r)), 2),
        'Best': round(float(np.max(r)), 1),
    })

dir_df = pd.DataFrame(dir_stats)
print('── Top 15 by film count ──')
print(dir_df.sort_values('Films', ascending=False).head(15).to_string(index=False))
print('\n── Top 15 by avg rating (min 5 films) ──')
print(dir_df[dir_df['Films']>=5].sort_values('Avg', ascending=False).head(15).to_string(index=False))

## 8. Revenue Analysis

Not all movies have gross revenue data. We drop NaN rows when working with revenue.

In [ ]:
# ── Revenue analysis ─────────────────────────────────────────────────────────
rev = df_clean.dropna(subset=['gross']).copy()
rev['gross_m'] = rev['gross'] / 1e6

print(f'Movies with revenue data: {len(rev):,} / {len(df_clean):,} ({len(rev)/len(df_clean)*100:.1f}%)')
print(f'\nRevenue stats (USD millions):')
print(f'  Mean:   ${np.mean(rev.gross_m.values):.1f}M')
print(f'  Median: ${np.median(rev.gross_m.values):.1f}M')
print(f'  Max:    ${np.max(rev.gross_m.values):.1f}M')

print('\n── Top 10 Highest-Grossing ──')
top_rev = rev.nlargest(10, 'gross')[['Movie Name','gross_m','rating']].copy()
top_rev.columns = ['Movie', 'Gross ($M)', 'IMDb Rating']
top_rev['Gross ($M)'] = top_rev['Gross ($M)'].round(1)
print(top_rev.to_string(index=False))

## 9. Metascore vs IMDb Rating — Critic vs Audience

`numpy.corrcoef` gives us the Pearson correlation coefficient. A value near 1.0 means critics and audiences agree perfectly.

In [ ]:
# ── Metascore correlation ─────────────────────────────────────────────────────
both = df_clean.dropna(subset=['meta', 'rating']).copy()
corr = np.corrcoef(both['meta'].values, both['rating'].values)[0, 1]
print(f'Pearson correlation (Metascore vs IMDb): {corr:.3f}')
print('→ Strong positive' if corr > 0.7 else '→ Moderate positive' if corr > 0.4 else '→ Weak correlation')

both['gap'] = both['meta'] / 10 - both['rating']
print('\n── Critics loved, audiences didn\'t (biggest positive gap) ──')
print(both.nlargest(8,'gap')[['Movie Name','rating','meta','gap']].to_string(index=False))
print('\n── Audiences loved, critics didn\'t (biggest negative gap) ──')
print(both.nsmallest(8,'gap')[['Movie Name','rating','meta','gap']].to_string(index=False))

## 10. Personal Data — My IMDb Ratings

The same EDA steps applied to my personal viewing history.

In [ ]:
# ── Load personal watched list ────────────────────────────────────────────────
my = pd.read_csv('Watched.csv', encoding='utf-8')
my.columns = my.columns.str.strip()

my['your_rating'] = pd.to_numeric(my['Your Rating'], errors='coerce')
my['imdb_rating'] = pd.to_numeric(my['IMDb Rating'], errors='coerce')
my['year']        = pd.to_numeric(my['Year'],        errors='coerce')
my['decade']      = (my['year'] // 10 * 10).astype('Int64')
my['genres_list'] = my['Genres'].fillna('').str.split(', ')

my_r   = my['your_rating'].dropna().values
imdb_r = my['imdb_rating'].dropna().values

print(f'Total rated:        {len(my):,}')
print(f'My avg rating:      {np.mean(my_r):.2f}')
print(f'IMDb avg (same):    {np.mean(imdb_r):.2f}')
print(f'My median:          {np.median(my_r):.1f}')
print(f'My std:             {np.std(my_r):.2f}')

paired = my.dropna(subset=['your_rating','imdb_rating'])
gap    = np.abs(paired['your_rating'].values - paired['imdb_rating'].values)
print(f'Contrarian index:   {(gap>=2).mean()*100:.1f}% of films rated ≥2 pts from IMDb')

my_genre_ctr = Counter(g for gl in my['genres_list'] for g in gl if g)
print('\nMy top genres:')
for g, c in my_genre_ctr.most_common(8):
    print(f'  {g:<15} {c:>4} films')

print('\nFilms watched by decade:')
for dec, cnt in my['decade'].value_counts().sort_index().items():
    print(f'  {dec}s   {cnt:>4}  {chr(9608) * (cnt // 3)}')

## 11. Merge Personal Data with Master Dataset

Join on the IMDb `tconst` identifier to find overlap and detect blindspots.

In [ ]:
# ── Merge on tconst ──────────────────────────────────────────────────────────
my['tconst']       = my['Const'].astype(str).str.strip()
df_clean['tconst'] = df_clean['tconst'].astype(str)

merged = my.merge(
    df_clean[['tconst','Movie Name','rating','votes','director','genres_list','runtime_mins']],
    on='tconst', how='inner'
)
print(f'My watched that exist in Top 10K: {len(merged):,} / {len(my):,}')

diff = (merged['your_rating'] - merged['rating']).dropna()
print(f'\nRating difference (mine - IMDb):')
print(f'  Mean diff:   {diff.mean():+.2f}  (positive = I rate higher)')
print(f'  Std diff:    {diff.std():.2f}')

# ── Blindspot detection ───────────────────────────────────────────────────────
top_genres  = [g for g, _ in my_genre_ctr.most_common(4)]
watched_ids = set(my['tconst'])

blindspots = df_clean[
    (df_clean['rating'] >= 7.5) &
    (df_clean['votes']  >= 50_000) &
    (~df_clean['tconst'].isin(watched_ids)) &
    (df_clean['genres_list'].apply(lambda gl: any(g in top_genres for g in gl)))
].sort_values('rating', ascending=False)

print(f'\nBlindspots (top genres: {top_genres[:3]}) — Top 15:')
print(blindspots[['Movie Name','rating','votes','Genre']].head(15).to_string(index=False))

## 12. Summary & Conclusions

| Observation | Finding |
|-------------|---------|
| Rating distribution | Right-skewed — most movies cluster between 6.0 and 7.5 |
| Top genre by count  | Drama dominates the dataset by a large margin |
| Critic vs audience  | Moderate positive correlation — they agree more than they diverge |
| Hidden gems         | Many high-rated films have fewer than 50K votes — underexposed |
| Personal patterns   | Contrarian index measures how often I deviate from the crowd |

**EDA Takeaways:**
- Always audit missing values before computing any statistics
- Multi-value columns require special handling (split + Counter accumulation)
- Median is more robust than mean for skewed distributions
- Correlation does not imply causation — higher budget ≠ better rating

---
*This notebook is the analytical foundation for the CineScope V2 interactive dashboard.*  
*The full dashboard (`index.html`) uses pre-computed JSON outputs from `preprocess/process.py`.*